In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import gc

In [2]:
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoConfig
from vllm import LLM

In [3]:
BASE_DIR = '/groups/chichengz/tnn/datasets/'

def get_gpu_memory(device=0):
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info(device)
    print(f'GPU memory used: {(total - free) / (1024**3):.2f} GB')

def load_reward_model(llm_dir, device="cuda:0"):
    tokenizer = AutoTokenizer.from_pretrained(llm_dir, trust_remote_code=True)
    config = AutoConfig.from_pretrained(llm_dir, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
        tokenizer.pad_token = tokenizer.eos_token
    config.pad_token_id = tokenizer.pad_token_id
    model = AutoModel.from_pretrained(
        llm_dir,
        config=config,
        device_map=device,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model

def load_generative_model(llm_dir, device="cuda:0"):
    model = AutoModelForCausalLM.from_pretrained(
        llm_dir,
        device_map=device,
        trust_remote_code=True,
    )
    model.eval()
    return model

### Reward Models

In [4]:
# rm_dir = BASE_DIR + "Llama3.1-8B-PRM-Deepseek-Data"
# rm_dir = BASE_DIR + "Skywork-Reward-V2-Llama3.2-3B"
# rm_dir = BASE_DIR + "Skywork-Reward-V2-Llama3.2-8B"
rm_dir = BASE_DIR + "Qwen2.5-Math-PRM-7B"

In [9]:
tokenizer, rw_tf = load_reward_model(rm_dir)
get_gpu_memory()

Loading weights:   0%|          | 0/342 [00:00<?, ?it/s]

Qwen2ForProcessRewardModel LOAD REPORT from: /groups/chichengz/tnn/datasets/Qwen2.5-Math-PRM-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPU memory used: 27.76 GB


### Generative Models

In [6]:
# llm_dir = BASE_DIR + "Llama3.2-1B-Instruct"
# llm_dir = BASE_DIR + "Llama3.2-3B-Instruct"
# llm_dir = BASE_DIR + "Qwen2.5-3B-Instruct"
llm_dir = BASE_DIR + "Qwen2.5-7B-Instruct"

In [7]:
# Transformers reports actual model memory; vLLM over-allocates for its KV cache.
llm_tf = load_generative_model(llm_dir)
get_gpu_memory()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

GPU memory used: 14.51 GB


In [8]:
get_gpu_memory()

GPU memory used: 14.51 GB
